# Agentic Cypher Retrieval Cycle

LLM_1 → generates Cypher for the question (prompt_1 already includes schema + question).
LLM_2 → reviews Cypher and returns requirements.
Requirements are appended to prompt_1 (section "Additional requirements"), then the cycle repeats.


In [35]:
from pathlib import Path

BASE_DIR = Path('/Users/i.sharganov/🟩🟩🟩/обучение/МАГА/мага_обучение/3_сем/Диплом/эксперимент/misc/agentic_cyph_retr_cycle_sandbox')
PROMPT_1_PATH = BASE_DIR / 'prompt_1_q1.txt'
PROMPT_2_PATH = BASE_DIR / 'prompt_2_reviewer.txt'

# QUESTION = 'Which documents do I need to bring to passport control in France?'
QUESTION = 'Which countries are mentioned in database?'


prompt_1_template = PROMPT_1_PATH.read_text(encoding='utf-8')
prompt_2_text = PROMPT_2_PATH.read_text(encoding='utf-8')

print('prompt_1:', PROMPT_1_PATH)
print('prompt_2:', PROMPT_2_PATH)
print('question:', QUESTION)


prompt_1: /Users/i.sharganov/🟩🟩🟩/обучение/МАГА/мага_обучение/3_сем/Диплом/эксперимент/misc/agentic_cyph_retr_cycle_sandbox/prompt_1_q1.txt
prompt_2: /Users/i.sharganov/🟩🟩🟩/обучение/МАГА/мага_обучение/3_сем/Диплом/эксперимент/misc/agentic_cyph_retr_cycle_sandbox/prompt_2_reviewer.txt
question: Which countries are mentioned in database?


In [36]:
import os

def fetch_schema_text():
    """Fetch schema from Neo4j and return a compact text summary."""
    try:
        from neo4j import GraphDatabase
    except Exception as e:
        raise RuntimeError('neo4j package not available; install it or replace fetch_schema_text') from e

    uri = os.getenv('NEO4J_URI', 'bolt://localhost:7688')
    user = 'neo4j'#os.getenv('NEO4J_USER', 'neo4j')
    password = '12345678'#os.getenv('NEO4J_PASSWORD', 'neo4j')

    driver = GraphDatabase.driver(uri, auth=(user, password))
    try:
        with driver.session() as session:
            record = session.run('CALL db.schema.visualization()').single()
            nodes = record['nodes']
            rels = record['relationships']

            labels = sorted({lbl for n in nodes for lbl in list(n.labels)})
            rel_lines = []
            for r in rels:
                start_labels = '/'.join(sorted(list(r.start_node.labels)))
                end_labels = '/'.join(sorted(list(r.end_node.labels)))
                rel_lines.append(f'(:{start_labels})-[:{r.type}]->(:{end_labels})')

            schema_text = 'NODE LABELS:\n' + '\n'.join(f'- {l}' for l in labels)
            schema_text += '\n\nRELATIONSHIPS:\n' + '\n'.join(f'- {l}' for l in sorted(set(rel_lines)))
            return schema_text
    finally:
        driver.close()

schema_text = fetch_schema_text()
print(schema_text[:1000])


NODE LABELS:
- AdditionalChecks
- Airport
- AtDesk
- BorderOfficer
- Cash
- CashAmount
- City
- Country
- DocumentCheck
- Employment
- Encounter
- HotelBookingDoc
- HotelPaymentCard
- HotelPaymentStatus
- Outcome
- PaymentCard
- Questioning
- ReturnTicket
- Stamping
- TravelInsurance
- Traveller
- VisitDuration
- VisitPurpose

RELATIONSHIPS:
- (:Airport)-[:locatedInCity]->(:City)
- (:City)-[:locatedInCountry]->(:Country)
- (:DocumentCheck)-[:requestedDocType]->(:Cash)
- (:DocumentCheck)-[:requestedDocType]->(:HotelBookingDoc)
- (:DocumentCheck)-[:requestedDocType]->(:PaymentCard)
- (:DocumentCheck)-[:requestedDocType]->(:ReturnTicket)
- (:DocumentCheck)-[:requestedDocType]->(:TravelInsurance)
- (:Encounter)-[:atAirport]->(:Airport)
- (:Encounter)-[:atCity]->(:City)
- (:Encounter)-[:controlStage]->(:AtDesk)
- (:Encounter)-[:hasDocumentCheck]->(:DocumentCheck)
- (:Encounter)-[:hasOfficer]->(:BorderOfficer)
- (:Encounter)-[:hasOutcome]->(:Outcome)
- (:Encounter)-[:hasQuestioning]->(:Quest

In [37]:
def build_prompt_1_with_context(prompt_template, schema_text, question):
    text = prompt_template
    replaced_schema = False
    replaced_question = False
    for ph in ('{SCHEMA_TEXT}', '{SCHEMA}'):
        if ph in text:
            text = text.replace(ph, schema_text)
            replaced_schema = True
    for ph in ('{USER_QUERY}', '{USER_QUESTION}'):
        if ph in text:
            text = text.replace(ph, question)
            replaced_question = True
    if not replaced_schema:
        text += '\n\nDATABASE SCHEMA\n' + schema_text
    if not replaced_question:
        text += '\n\nQUESTION\n' + question
    text += '\n\nGenerate only Cypher (no explanations).\n'
    return text

prompt_1_text = build_prompt_1_with_context(prompt_1_template, schema_text, QUESTION)
PROMPT_1_PATH.write_text(prompt_1_text, encoding='utf-8')
print('prompt_1 updated with schema + question')


prompt_1 updated with schema + question


In [38]:
import json
import os
import requests

def call_llm(prompt, model=None, temperature=0):
    """DeepSeek chat completion call. Requires DEEPSEEK_API_KEY."""
    api_key = 'sk-8da8031877dc4c39ae8d7b624c70f01f'
    if not api_key:
        raise RuntimeError('DEEPSEEK_API_KEY is not set')
    model = model or os.getenv('DEEPSEEK_MODEL', 'deepseek-chat')
    url = os.getenv('DEEPSEEK_BASE_URL', 'https://api.deepseek.com/v1/chat/completions')
    headers = {
        'Authorization': f'Bearer {api_key}',
        'Content-Type': 'application/json'
    }
    payload = {
        'model': model,
        'messages': [{'role': 'user', 'content': prompt}],
        'temperature': temperature
    }
    resp = requests.post(url, headers=headers, json=payload, timeout=60)
    resp.raise_for_status()
    data = resp.json()
    return data['choices'][0]['message']['content']


In [39]:
import json
import re

def _extract_json(text):
    if not text:
        raise ValueError('Empty response from LLM')
    s = text.strip()
    # strip code fences if present
    if s.startswith('```'):
        s = re.sub(r'^```[a-zA-Z0-9_-]*\n', '', s)
        s = re.sub(r'\n```$', '', s)
        s = s.strip()
    # try full parse
    try:
        return json.loads(s)
    except Exception:
        pass
    # try to find first JSON object
    m = re.search(r'\{.*\}', s, re.S)
    if m:
        return json.loads(m.group(0))
    raise ValueError('Failed to parse JSON from LLM output')

def llm1_generate_cypher(prompt_1_text, model=None):
    return call_llm(prompt_1_text, model=model)

def llm2_review(question, cypher, prompt_1_text, prompt_2_text, model=None):
    prompt = (
        f"{prompt_2_text}\n\n"
        f"QUESTION:\n{question}\n\n"
        f"PROMPT_1:\n{prompt_1_text}\n\n"
        f"CYPHER:\n{cypher}\n"
    )
    out = call_llm(prompt, model=model)
    try:
        return _extract_json(out)
    except Exception as e:
        # Retry once with stronger instruction
        retry_prompt = prompt + '\n\nRETURN ONLY STRICT JSON. NO TEXT.'
        out2 = call_llm(retry_prompt, model=model)
        return _extract_json(out2)


In [40]:
def ensure_additional_section(text):
    if 'Additional requirements' not in text:
        return text + '\n\nAdditional requirements:\n'
    return text

def append_requirements(text, reqs):
    text = ensure_additional_section(text)
    existing = set()
    for line in text.splitlines():
        line = line.strip()
        if line.startswith('- '):
            existing.add(line[2:])
    new_reqs = [r for r in reqs if r and r not in existing]
    if not new_reqs:
        return text
    return text + '\n' + '\n'.join(f'- {r}' for r in new_reqs)


In [41]:
def run_agentic_cycle(question, max_iters=3, model_llm1=None, model_llm2=None):
    prompt_1_template = PROMPT_1_PATH.read_text(encoding='utf-8')
    prompt_2 = PROMPT_2_PATH.read_text(encoding='utf-8')

    schema_text = fetch_schema_text()
    prompt_1 = build_prompt_1_with_context(prompt_1_template, schema_text, question)
    PROMPT_1_PATH.write_text(prompt_1, encoding='utf-8')

    history = []
    for i in range(max_iters):
        cypher = llm1_generate_cypher(prompt_1, model=model_llm1)
        review = llm2_review(question, cypher, prompt_1, prompt_2, model=model_llm2)
        history.append({'iter': i+1, 'cypher': cypher, 'review': review})

        if review.get('review_status') == 'OK':
            return {
                'status': 'OK',
                'cypher': cypher,
                'review': review,
                'prompt_1_final': prompt_1,
                'history': history,
            }

        reqs = review.get('requirements_to_append', [])
        prompt_1 = append_requirements(prompt_1, reqs)
        PROMPT_1_PATH.write_text(prompt_1, encoding='utf-8')

    return {
        'status': 'NEEDS_MANUAL',
        'history': history,
        'prompt_1_final': prompt_1,
    }


In [42]:
# Example usage
result = run_agentic_cycle(QUESTION)
result


{'status': 'NEEDS_MANUAL',
 'history': [{'iter': 1,
   'cypher': 'MATCH (c:Country)\nRETURN DISTINCT c\nLIMIT 200',
   'review': {'review_status': 'NEEDS_FIX',
    'blocking_issues': ["The query returns Country nodes but doesn't answer which countries are 'mentioned in database'. The question asks for countries mentioned anywhere in the database, which could include countries referenced through relationships (e.g., via City or Airport nodes) even if no direct Country node exists. The current query only returns nodes with the Country label, potentially missing countries mentioned indirectly."],
    'requirements_to_append': ['The query must return all distinct country names mentioned anywhere in the database, including countries referenced through relationships like (:City)-[:locatedInCountry]->(:Country) or (:Encounter)-[:atCity]->(:City)-[:locatedInCountry]->(:Country). Use DISTINCT to avoid duplicates and include a LIMIT if results could be large.']}},
  {'iter': 2,
   'cypher': 'MAT